# Cell 1 — Imports & Data

In [ ]:
import os
import cv2 as c
import torch as t
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from collections import Counter

comp_train    = "/kaggle/input/datasets/abdelrahmanbakrg/emotion-data/Training_data/Training_data"
comp_test     = "/kaggle/input/datasets/abdelrahmanbakrg/emotion-data/test/test"
ferplus_train = "/kaggle/input/datasets/arnabkumarroy02/ferplus/train"
ferplus_val   = "/kaggle/input/datasets/arnabkumarroy02/ferplus/validation"

device = t.device("cuda" if t.cuda.is_available() else "cpu")

IMG_SIZE      = 128
VALID_CLASSES = {"fear", "happy", "disgust", "sad", "surprise", "angry"}
label_map     = {cls: i for i, cls in enumerate(sorted(VALID_CLASSES))}
idx_to_label  = {v: k for k, v in label_map.items()}

# الـ transforms زي الأصل بالظبط (grayscale)
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomCrop(IMG_SIZE, padding=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

def load_images(paths, filter_classes=None):
    images, labels = [], []
    for path in paths:
        for cls_name in os.listdir(path):
            cls_normalized = "surprise" if cls_name == "suprise" else cls_name
            if filter_classes and cls_normalized not in filter_classes:
                continue
            cls_path = os.path.join(path, cls_name)
            if not os.path.isdir(cls_path):
                continue
            for img_file in os.listdir(cls_path):
                if not img_file.endswith(('.jpg', '.png')):
                    continue
                img = c.imread(os.path.join(cls_path, img_file), c.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                images.append(img)
                labels.append(label_map[cls_normalized])
    return images, labels

def load_test_images(path):
    images, filenames = [], []
    for img_file in os.listdir(path):
        if not img_file.endswith('.jpg'):
            continue
        img = c.imread(os.path.join(path, img_file), c.IMREAD_GRAYSCALE)
        if img is None:
            continue
        images.append(img)
        filenames.append(img_file)
    return images, filenames

class EmotionDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img   = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, t.tensor(label, dtype=t.long)

class TestDataset(Dataset):
    def __init__(self, images, transform=None):
        self.images    = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return img

print("Loading training data...")
train_imgs, train_lbls = load_images(
    paths=[comp_train, ferplus_train, ferplus_val],
    filter_classes=VALID_CLASSES
)

print("Loading test data...")
test_imgs, test_filenames = load_test_images(comp_test)

print(f"Total train samples : {len(train_imgs)}")
print(f"Class distribution  : {Counter(train_lbls)}")
print(f"Total test samples  : {len(test_imgs)}")
print(f"Label map           : {label_map}")

class_counts  = Counter(train_lbls)
total_samples = len(train_lbls)
class_weights = t.tensor([
    total_samples / (len(class_counts) * class_counts[i])
    for i in range(len(label_map))
], dtype=t.float32).to(device)
print(f"Class weights       : {class_weights}")

full_dataset = EmotionDataset(train_imgs, train_lbls, transform=train_transform)
val_size     = int(0.15 * len(full_dataset))
train_size   = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size],
                                 generator=t.Generator().manual_seed(42))

val_ds = EmotionDataset(
    [train_imgs[i] for i in val_ds.indices],
    [train_lbls[i] for i in val_ds.indices],
    transform=test_transform
)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(TestDataset(test_imgs, transform=test_transform), batch_size=32)

print(f"\nTrain batches : {len(train_dl)}")
print(f"Val batches   : {len(val_dl)}")
print(f"Test batches  : {len(test_dl)}")

# Cell 2 — VGG Backbone (من الصفر)

نفس الـ VGGNet الأصلي بالظبط بس بدون الـ classifier (بس الـ features)  
عشان نستخدمه كـ branch في الـ Hybrid ونجيب منه feature map

In [ ]:
class VGGBackbone(nn.Module):
    """
    نفس بلوكات الـ VGGNet الأصلية بالظبط،
    بس من غير الـ classifier الأخير.
    بياخد صورة grayscale (1 channel)
    ويطلع feature map حجمها (B, 512, H/32, W/32)
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: (1, 128, 128) → (64, 64, 64)
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),

            # Block 2: (64, 64, 64) → (128, 32, 32)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),

            # Block 3: (128, 32, 32) → (256, 16, 16)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),

            # Block 4: (256, 16, 16) → (512, 8, 8)
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),

            # Block 5: (512, 8, 8) → (512, 4, 4)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.15),
        )
        self.out_channels = 512

    def forward(self, x):
        # x shape: (B, 1, 128, 128)
        return self.features(x)  # output: (B, 512, 4, 4)


# تجربة سريعة للتأكد إن الأبعاد صح
dummy = t.zeros(2, 1, 128, 128)
vgg_test = VGGBackbone()
out = vgg_test(dummy)
print(f"VGG output shape : {out.shape}")  # expected: (2, 512, 4, 4)

# Cell 3 — MobileViT Backbone

MobileViT بياخد 3 channels، بس صورنا grayscale (1 channel)  
الحل: نضيف **Channel Adapter** (Conv 1→3) قبل MobileViT  
وبنستخدم `pretrained=True` عشان ياخد فائدة الـ pretraining على ImageNet

In [ ]:
import timm

class MobileViTBackbone(nn.Module):
    """
    MobileViT-Small مع adapter للـ grayscale input.

    المشكلة: MobileViT اتدرب على RGB (3 channels)
    الحل:    Conv2d(1 → 3) بتوسع الـ channel قبل ما الصورة تدخل MobileViT

    الـ output: feature map حجمها (B, 640, H/32, W/32)
    """
    def __init__(self):
        super().__init__()

        # ① Channel Adapter: grayscale (1ch) → pseudo-RGB (3ch)
        #    kernel_size=1 يعني بس بيغير القنوات من غير ما يمس الـ spatial info
        self.channel_adapter = nn.Sequential(
            nn.Conv2d(1, 3, kernel_size=1, bias=False),
            nn.BatchNorm2d(3),
        )

        # ② MobileViT-Small بدون الـ classifier head
        #    features_only=True → بيرجع intermediate feature maps
        #    pretrained=True    → أوزان مدربة مسبقاً على ImageNet
        self.backbone = timm.create_model(
            'mobilevit_s',
            pretrained=True,
            features_only=True,
        )

        # عدد القنوات في آخر feature map في mobilevit_s هو 640
        self.out_channels = self.backbone.feature_info[-1]['num_chs']  # 640

    def forward(self, x):
        # x shape: (B, 1, 128, 128)
        x = self.channel_adapter(x)     # → (B, 3, 128, 128)
        features = self.backbone(x)     # list of feature maps
        return features[-1]             # آخر feature map: (B, 640, 4, 4)


# تجربة للتأكد
dummy = t.zeros(2, 1, 128, 128)
vit_test = MobileViTBackbone()
out = vit_test(dummy)
print(f"MobileViT output shape : {out.shape}")  # expected: (2, 640, 4, 4)
print(f"MobileViT out_channels : {vit_test.out_channels}")

# Cell 4 — Hybrid Model (VGG + MobileViT)

بنربط الـ branch دول مع بعض:
```
Input (B, 1, 128, 128)
    ├──→ VGGBackbone     → (B, 512, 4, 4)   [local features]
    └──→ MobileViTBackbone → (B, 640, 4, 4) [global features]
              ↓
         torch.cat dim=1  → (B, 1152, 4, 4)
              ↓
         Fusion Conv 1×1  → (B, 512, 4, 4)
              ↓
         AdaptiveAvgPool  → (B, 512, 1, 1)
              ↓
         Classifier       → (B, 6)
```

In [ ]:
class MobileViT_VGG_Hybrid(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()

        # ① الفرعين (branches)
        self.vgg_branch = VGGBackbone()        # output: 512 channels
        self.vit_branch = MobileViTBackbone()  # output: 640 channels

        vgg_ch  = self.vgg_branch.out_channels   # 512
        vit_ch  = self.vit_branch.out_channels   # 640
        fused   = vgg_ch + vit_ch                # 1152

        # ② Fusion layer: بيمزج الـ features من الفرعين
        #    Conv2d 1×1 → بتمزج القنوات مع بعض من غير ما تأثر على الحجم المكاني
        self.fusion = nn.Sequential(
            nn.Conv2d(fused, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
        )

        # ③ Classifier: نفس تصميم الـ VGG الأصلي
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # (B, 512, 4, 4) → (B, 512, 1, 1)
            nn.Flatten(),                  # (B, 512)
            nn.Linear(512, 256),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        # x: (B, 1, 128, 128)

        vgg_feat = self.vgg_branch(x)    # (B, 512, 4, 4)
        vit_feat = self.vit_branch(x)    # (B, 640, 4, 4)

        # لو في فرق في الحجم المكاني بسبب اختلاف الـ stride، بنعدله
        if vit_feat.shape[2:] != vgg_feat.shape[2:]:
            vit_feat = nn.functional.interpolate(
                vit_feat,
                size=vgg_feat.shape[2:],
                mode='bilinear',
                align_corners=False
            )

        fused = t.cat([vgg_feat, vit_feat], dim=1)  # (B, 1152, 4, 4)
        fused = self.fusion(fused)                   # (B, 512, 4, 4)
        return self.classifier(fused)                # (B, 6)


# Kaiming initialization للـ VGG والـ fusion من الصفر
def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight)
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)

model = MobileViT_VGG_Hybrid(num_classes=6).to(device)

# بنطبق init بس على VGG والـ fusion، مش على MobileViT (هو عنده أوزانه)
model.vgg_branch.apply(init_weights)
model.fusion.apply(init_weights)
model.classifier.apply(init_weights)

# تجربة forward pass كاملة
dummy = t.zeros(2, 1, 128, 128).to(device)
with t.no_grad():
    out = model(dummy)
print(f"Final output shape   : {out.shape}")  # expected: (2, 6)

total_params = sum(p.numel() for p in model.parameters())
vgg_params   = sum(p.numel() for p in model.vgg_branch.parameters())
vit_params   = sum(p.numel() for p in model.vit_branch.parameters())
print(f"Total parameters     : {total_params:,}")
print(f"  VGG branch         : {vgg_params:,}")
print(f"  MobileViT branch   : {vit_params:,}")

# Cell 5 — Training

نفس loop التدريب الأصلي، بس بـ learning rates مختلفة:
- **MobileViT**: lr صغيرة (1e-4) عشان اتدرب مسبقاً
- **VGG + Fusion + Classifier**: lr أكبر (3e-4) عشان بيتدربوا من الصفر

In [ ]:
import pandas as pd
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Differential learning rates
# MobileViT: lr صغيرة عشان pretrained
# الباقي: lr أكبر عشان من الصفر
optimizer = AdamW([
    {'params': model.vit_branch.backbone.parameters(),        'lr': 1e-4, 'weight_decay': 1e-2},
    {'params': model.vit_branch.channel_adapter.parameters(), 'lr': 3e-4, 'weight_decay': 1e-2},
    {'params': model.vgg_branch.parameters(),                 'lr': 3e-4, 'weight_decay': 1e-2},
    {'params': model.fusion.parameters(),                     'lr': 3e-4, 'weight_decay': 1e-2},
    {'params': model.classifier.parameters(),                 'lr': 3e-4, 'weight_decay': 1e-2},
])

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler    = t.amp.GradScaler('cuda')

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with t.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            with t.amp.autocast('cuda'):
                outputs = model(imgs)
                loss    = criterion(outputs, lbls)
            total_loss += loss.item() * imgs.size(0)
            correct    += outputs.argmax(dim=1).eq(lbls).sum().item()
            total      += lbls.size(0)
    return total_loss / total, 100 * correct / total

def check_overfitting(train_acc, val_acc, train_loss, val_loss, gap_threshold=10.0):
    gap = train_acc - val_acc
    print(f"  ↳ Acc Gap (train - val): {gap:.2f}%", end="  ")
    if gap > gap_threshold:
        print(f"⚠️  Overfitting! gap > {gap_threshold}%")
    elif val_loss > train_loss * 1.3:
        print("⚠️  Overfitting! val_loss >> train_loss")
    else:
        print("✅ OK")

EPOCHS           = 60
PATIENCE         = 10
best_val_acc     = 0
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for imgs, lbls in train_dl:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()

        with t.amp.autocast('cuda'):
            outputs = model(imgs)
            loss    = criterion(outputs, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        correct    += outputs.argmax(dim=1).eq(lbls).sum().item()
        total      += lbls.size(0)

    train_loss_avg    = total_loss / len(train_dl)
    train_acc         = 100 * correct / total
    val_loss, val_acc = evaluate(model, val_dl)

    scheduler.step(val_acc)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}]  "
          f"Train Loss: {train_loss_avg:.4f}  Train Acc: {train_acc:.2f}%  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%")

    check_overfitting(train_acc, val_acc, train_loss_avg, val_loss)

    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        t.save(model.state_dict(), "best_hybrid.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stop  |  best val acc: {best_val_acc:.2f}%")
            break

print(f"\nBest Val Acc: {best_val_acc:.2f}%")

# Cell 6 — Inference + Submission

In [ ]:
import re

model.load_state_dict(t.load("best_hybrid.pth"))

tta_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

TTA_RUNS = 5
model.eval()

all_preds = []
with t.no_grad():
    for imgs in test_dl:
        imgs  = imgs.to(device)
        probs = t.zeros(imgs.size(0), 6).to(device)
        for _ in range(TTA_RUNS):
            with t.amp.autocast('cuda'):
                probs += t.softmax(model(imgs), dim=1)
        all_preds.extend(probs.argmax(dim=1).cpu().tolist())

def filename_to_id(path):
    match = re.search(r'([^/\\]+)(?=\.\w+$)', path)
    return match.group(1) if match else path

ids = [filename_to_id(f) for f in test_filenames]

df = pd.DataFrame({
    "ID":    ids,
    "Label": [idx_to_label[p] for p in all_preds]
})

df.to_csv("hybrid_submission.csv", index=False)
print(df.head(10))
print(f"\nSaved hybrid_submission.csv → {len(df)} rows")
print(f"Best Val Acc: {best_val_acc:.2f}%")